# Sistema de Verificação de Decolagem
Projeto baseado em telemetria para validação de condições de lançamento.

In [ ]:
#Model

class SistemaDecolagemModel:
    def __init__(self):
        self.limites = {
            "temp_int": (18, 27),
            "temp_ext": (9.5, 35),
            "pressao": (350, 400),
            #Deixei a sugestão em kPa, equivalente a 51 a 57 psia.
            "energia_min": 90.5,
            "integridade_ok": 1
        }

    def validar_decolagem(self, dados):
        erros = []

        if not (self.limites["temp_int"][0] <= dados['temp_int'] <= self.limites["temp_int"][1]):
            erros.append(f"Temperatura Interna diferente do intervalo seguro ({dados['temp_int']}ºC)")

        if not (self.limites["temp_ext"][0] <= dados['temp_ext'] <= self.limites["temp_ext"][1]):
            erros.append(f"Temperatura Externa diferente do intervalo seguro ({dados['temp_ext']}ºC)")

        if not (self.limites["pressao"][0] <= dados['pressao_tanque'] <= self.limites["pressao"][1]):
            erros.append(f"Pressão dos Tanques fora do limite ({dados['pressao_tanque']})")

        if dados['nivel_energia'] < self.limites["energia_min"]:
            erros.append(f"Energia insuficiente para decolagem ({dados['nivel_energia']}%)")

        if dados['integridade'] != self.limites["integridade_ok"]:
            erros.append("Revise a integridade da estrutura")

        if not dados['modulos_ok']:
            erros.append("Falha grave nos módulos críticos")

        return erros

In [ ]:
#View

class TerminalView:
    @staticmethod
    def coletar_dados():
        print("\n" + "=" * 30)
        print("  SISTEMA DE VERIFICAÇÃO - DECOLAGEM")
        print("=" * 30)
        try:
            return {
                'temp_int': float(input("Temperatura Interna (ºC): ")),
                'temp_ext': float(input("Temperatura Externa (ºC): ")),
                'pressao_tanque': float(input("Pressão dos Tanques (kPa): ")),
                'nivel_energia': float(input("Nível de Energia (%): ")),
                'integridade': int(input("Integridade (1=OK / 0=FALHA): ")),
                'modulos_ok': input("Módulos Críticos OK? (s/n): ").lower().startswith('s')
            }
        except ValueError:
            return None

    @staticmethod
    def exibir_resultado(sucesso, falhas=None):
        if sucesso:
            print("\n" + "#" * 25)
            print("  PRONTO PARA DECOLAR")
            print("#" * 25)
        else:
            print("\n" + "!" * 25)
            print("  DECOLAGEM ABORTADA")
            print("!" * 25)
            if falhas:
                print("\nMOTIVOS DO CANCELAMENTO:")
                for erro in falhas:
                    print(f" -> {erro}")

In [ ]:
#Controller

class DecolagemController:
    def __init__(self):
        self.model = SistemaDecolagemModel()
        self.view = TerminalView()

    def iniciar(self):
        dados = self.view.coletar_dados()

        if dados is None:
            self.view.exibir_resultado(False, ["Dados de entrada inválidos (insira números onde solicitado)."])
            return

        lista_erros = self.model.validar_decolagem(dados)

        decolagem_autorizada = len(lista_erros) == 0

        self.view.exibir_resultado(decolagem_autorizada, lista_erros)

In [ ]:
#Main - Executar

app = DecolagemController()
app.iniciar()